<a href="https://colab.research.google.com/github/Navya449/Victim_detection/blob/main/Victim_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install librosa
!pip install soundfile
!pip install tensorflow
!pip install scikit-learn

In [ ]:
import os
import librosa
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive
import os
import shutil

# Attempt to clean up the mount point if it's not actually mounted
path = '/content/drive'
if os.path.exists(path) and not os.path.ismount(path):
    print(f'Removing existing local directory {path} to allow mounting...')
    shutil.rmtree(path)

try:
    drive.mount(path, force_remount=True)
except Exception as e:
    print(f'Mount failed: {e}')

In [ ]:
import os

project_path = "/content/drive/MyDrive/Victim_Voice_Project"

print(os.path.exists(project_path))

In [ ]:
!wget https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz

In [ ]:
!tar -xzf speech_commands_v0.02.tar.gz

In [ ]:
!ls

In [ ]:
import os

count = 0

for folder in os.listdir():
    if os.path.isdir(folder):
        for file in os.listdir(folder):
            if file.endswith(".wav"):
                count += 1

print("Total speech files:", count)

In [ ]:
import os

noise_folder = "_background_noise_"

print(os.listdir(noise_folder))

In [ ]:
import os

os.makedirs("dataset/voice", exist_ok=True)
os.makedirs("dataset/noise", exist_ok=True)

print("Folders created successfully!")

In [ ]:
import os

base_path = "/content/drive/MyDrive/Victim_Voice_Project"

os.makedirs(os.path.join(base_path, "dataset", "voice"), exist_ok=True)
os.makedirs(os.path.join(base_path, "dataset", "noise"), exist_ok=True)

print("Folders created successfully!")

In [ ]:
import os
import shutil

commands = [
    "yes",
    "no",
    "up",
    "down",
    "left",
    "right",
    "go",
    "stop",
    "on",
    "off"
]

count = 0

for command in commands:
    for file in os.listdir(command):
        if file.endswith(".wav"):
            src = os.path.join(command, file)
            dst = os.path.join("dataset/voice", command + "_" + file)
            shutil.copy(src, dst)
            count += 1

print("Voice files copied:", count)

In [ ]:
import librosa
import soundfile as sf
import os

noise_folder = "_background_noise_"
output_folder = "dataset/noise"

sample_rate = 16000
clip_length = sample_rate   # 1 second

clip_number = 0

for file in os.listdir(noise_folder):

    if not file.endswith(".wav"):
        continue

    audio, sr = librosa.load(
        os.path.join(noise_folder, file),
        sr=sample_rate
    )

    total_clips = len(audio) // clip_length

    for i in range(total_clips):

        start = i * clip_length
        end = start + clip_length

        clip = audio[start:end]

        sf.write(
            os.path.join(output_folder, f"noise_{clip_number}.wav"),
            clip,
            sample_rate
        )

        clip_number += 1

print("Noise clips created:", clip_number)

In [ ]:
import os

voice_path = "dataset/voice"
noise_path = "dataset/noise"

voice_count = len([f for f in os.listdir(voice_path) if f.endswith(".wav")])
noise_count = len([f for f in os.listdir(noise_path) if f.endswith(".wav")])

print("Voice files :", voice_count)
print("Noise files :", noise_count)

In [ ]:
import os

print(os.listdir("dataset"))

In [ ]:
import os
import librosa
import numpy as np
import random
from tqdm import tqdm

In [ ]:
X = []
y = []

In [ ]:
voice_folder = "dataset/voice"

voice_files = [
    f for f in os.listdir(voice_folder)
    if f.endswith(".wav")
]

random.seed(42)

voice_files = random.sample(voice_files, 398)

print(len(voice_files))

In [ ]:
for file in tqdm(voice_files):

    path = os.path.join(voice_folder, file)

    signal, sr = librosa.load(path, sr=16000)

    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=40
    )

    if mfcc.shape[1] < 32:
        mfcc = np.pad(
            mfcc,
            ((0,0),(0,32-mfcc.shape[1])),
            mode="constant"
        )
    else:
        mfcc = mfcc[:, :32]

    X.append(mfcc)
    y.append("voice")

In [ ]:
noise_folder = "dataset/noise"

noise_files = [
    f for f in os.listdir(noise_folder)
    if f.endswith(".wav")
]

print(len(noise_files))

In [ ]:
for file in tqdm(noise_files):

    path = os.path.join(noise_folder, file)

    signal, sr = librosa.load(path, sr=16000)

    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=40
    )

    if mfcc.shape[1] < 32:
        mfcc = np.pad(
            mfcc,
            ((0,0),(0,32-mfcc.shape[1])),
            mode="constant"
        )
    else:
        mfcc = mfcc[:, :32]

    X.append(mfcc)
    y.append("noise")

In [ ]:
X = np.array(X)
y = np.array(y)

print("Features :", X.shape)
print("Labels :", y.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print(encoder.classes_)
print(y[:10])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples :", X_train.shape)
print("Testing Samples :", X_test.shape)

In [ ]:
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(X_train.shape)
print(X_test.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

model = Sequential()

model.add(Conv2D(
    32,
    (3,3),
    activation='relu',
    input_shape=(40,32,1)
))

model.add(MaxPooling2D((2,2)))

model.add(Conv2D(
    64,
    (3,3),
    activation='relu'
))

model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(
    128,
    activation='relu'
))

model.add(Dropout(0.3))

model.add(Dense(
    2,
    activation='softmax'
))

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model Compiled Successfully")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "victim_voice_model.keras",
    monitor="val_accuracy",
    save_best_only=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
import numpy as np

y_pred = model.predict(X_test)

y_pred = np.argmax(y_pred, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,5))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()

plt.xticks([0,1], ["Noise","Voice"])
plt.yticks([0,1], ["Noise","Voice"])

plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j],
                 ha='center',
                 va='center',
                 color='black')

plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Noise","Voice"]
))

# New Section

In [ ]:
model.save("victim_voice_model.keras")

print("Model Saved Successfully")

In [ ]:
import json

labels = {
    0: "noise",
    1: "voice"
}

with open("labels.json", "w") as f:
    json.dump(labels, f)

print("Labels Saved")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("victim_voice_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TensorFlow Lite Model Saved")

Verification

In [ ]:
import os

print(os.path.exists("test_case.wav"))

True


In [ ]:
import librosa

signal, sr = librosa.load("test_case.wav", sr=16000)

print("Sample Rate:", sr)
print("Audio Length:", len(signal))

/tmp/ipykernel_5625/3706924954.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
  signal, sr = librosa.load("test_case.wav", sr=16000)


Sample Rate: 16000
Audio Length: 45398


In [ ]:
import numpy as np

mfcc = librosa.feature.mfcc(
    y=signal,
    sr=sr,
    n_mfcc=40
)

print("Original Shape:", mfcc.shape)

Original Shape: (40, 89)


In [ ]:
if mfcc.shape[1] < 32:
    mfcc = np.pad(
        mfcc,
        ((0,0),(0,32-mfcc.shape[1])),
        mode="constant"
    )
else:
    mfcc = mfcc[:, :32]

print("Resized Shape:", mfcc.shape)

Resized Shape: (40, 32)


In [ ]:
mfcc = mfcc.reshape(1,40,32,1)

In [ ]:
import os

print(os.listdir())

['.config', '.ipynb_checkpoints', 'drive', 'test_case.wav', 'sample_data']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
